In [14]:
import torch
import os
import sys
import json


project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

from gpt_lib import (
    Config,
    create_config,
    Generator,
    build_training_diagnostics_report,
    build_model,
    build_trainer,
    create_epoch_checkpoint_callback,
    format_duration,
    format_long_context_results,
    load_compatible_checkpoint,
    prepare_data,
    run_long_context_evaluation,
    save_training_checkpoint,
    SentencePieceTokenizer
)

In [15]:
def print_data_summary(data):
    print(f"Data tokens: {len(data.tokens)}")
    print(f"Train tokens: {len(data.train_tokens)}")
    print(f"Validation tokens: {len(data.val_tokens)}")
    print(f"Vocab size: {data.vocab_size}")
    print(f"Train batches: {len(data.train_loader)}")
    if data.val_loader is not None:
        print(f"Validation batches: {len(data.val_loader)}")

In [16]:
old_model_settings = {
    'embed_dim': 64, 
    'block_size': 128, 
    'batch_size': 32, 
    'epochs': 10, 
    'learning_rate': 0.00025, 
    'weight_decay': 0.1, 
    'dropout': 0.1, 
    'grad_clip': 1.0, 
    'num_blocks': 3, 
    'domain_data_path': "../data/data.txt", 
    'domain_data_repeats': 3, 
    'tokenizer_type': 'sentencepiece', 
    'sentencepiece_model_type': 'bpe', 
    'sentencepiece_character_coverage': 1.0, 
    'max_vocab': 10000, 
    'max_data_size': 1000000, 
    'device': 'cpu', 
    'validation_split': 0.1, 
    'early_stopping_patience': 2, 
    'early_stopping_min_delta': 0.0001, 
    'restore_best_model': True, 
    'seed': 42, 
    'diagnostic_top_k': 5, 
    'concept_benchmark_top_k': 10, 
    'diagnostic_sample_tokens': 60, 
    'tokenizer_rare_threshold': 2, 
    'training_log_interval': 50, 
    'run_long_context_evaluation': False,
    'long_context_block_sizes': [32, 64, 128]
    }

In [17]:
cfg = create_config(
    **old_model_settings,
    model_path = "../output/mini_gpt-fine_2.pth",
    data_path = "../data/bookcorpus_diverse.txt",
    diagnostic_prompts = ["python is a ", "javascript is a"],
    use_checkpoint_tokenizer=True
)
cfg.to_dict()

{'embed_dim': 64,
 'block_size': 128,
 'batch_size': 32,
 'epochs': 10,
 'learning_rate': 0.00025,
 'weight_decay': 0.1,
 'dropout': 0.1,
 'grad_clip': 1.0,
 'num_blocks': 3,
 'model_path': '../output/mini_gpt-fine_2.pth',
 'data_path': '../data/bookcorpus_diverse.txt',
 'domain_data_path': '../data/data.txt',
 'domain_data_repeats': 3,
 'tokenizer_type': 'sentencepiece',
 'sentencepiece_model_type': 'bpe',
 'sentencepiece_character_coverage': 1.0,
 'tokenizer_max_line_length': 4000,
 'max_vocab': 10000,
 'max_data_size': 1000000,
 'device': 'cpu',
 'validation_split': 0.1,
 'early_stopping_patience': 2,
 'early_stopping_min_delta': 0.0001,
 'restore_best_model': True,
 'seed': 42,
 'diagnostic_top_k': 5,
 'concept_benchmark_top_k': 10,
 'diagnostic_prompts': ['python is a ', 'javascript is a'],
 'diagnostic_sample_tokens': 60,
 'tokenizer_rare_threshold': 2,
 'training_log_interval': 50,
 'run_long_context_evaluation': False,
 'use_checkpoint_tokenizer': True,
 'long_context_block_siz

In [18]:

with open("tokenizer.json", "r", encoding="utf-8") as f:
    text = f.read()

tkz_data = json.loads(text)
old_tkz = SentencePieceTokenizer.from_json(tkz_data)
data = prepare_data(cfg, old_tokenizer= old_tkz)
print_data_summary(data)

tokenizer is old!
Data tokens: 1000000
Train tokens: 805864
Validation tokens: 89421
Vocab size: 10000
Train batches: 47636
Validation batches: 5282


In [19]:
# tokenizer = data.tokenizer.to_json()

In [20]:
model = build_model(cfg, data.vocab_size)
print(f"Parameters: {model.get_num_parameters():,}")

Parameters: 1,435,792


In [21]:
trainer = build_trainer(model, cfg)
checkpoint_loaded = load_compatible_checkpoint(trainer, cfg, data.vocab_size)

same_vocab, same_shape, same_tokenizer, same_training_strategy True True True True
[OK] Model loaded from ../output/mini_gpt-fine_2.pth at epoch 7


In [22]:
checkpoint_loaded

True

In [23]:
if not checkpoint_loaded:
        print("\nNo improved checkpoint found; starting training.")
checkpoint_callback = create_epoch_checkpoint_callback(
    cfg,
    data.tokenizer,
    data.vocab_size,
)
trainer.train(
    data.train_loader,
    cfg.epochs,
    val_loader=data.val_loader,
    early_stopping_patience=cfg.early_stopping_patience,
    checkpoint_callback=checkpoint_callback,
)
save_training_checkpoint(trainer, cfg, data.tokenizer, data.vocab_size)


[OK] Resuming training from epoch 8/10.
Epoch 8/10 | Batch 50/47636 | Loss = 1.1410 | Elapsed = 32.9s | ETA = 31280.5s
Epoch 8/10 | Batch 100/47636 | Loss = 0.6664 | Elapsed = 69.5s | ETA = 33018.1s
Epoch 8/10 | Batch 150/47636 | Loss = 0.5675 | Elapsed = 103.6s | ETA = 32810.2s
Epoch 8/10 | Batch 200/47636 | Loss = 0.5624 | Elapsed = 142.7s | ETA = 33850.9s
Epoch 8/10 | Batch 250/47636 | Loss = 0.5491 | Elapsed = 183.6s | ETA = 34793.0s
Epoch 8/10 | Batch 300/47636 | Loss = 0.5444 | Elapsed = 222.7s | ETA = 35138.3s
Epoch 8/10 | Batch 350/47636 | Loss = 0.5132 | Elapsed = 258.5s | ETA = 34921.8s
Epoch 8/10 | Batch 400/47636 | Loss = 0.5515 | Elapsed = 295.5s | ETA = 34895.2s
Epoch 8/10 | Batch 450/47636 | Loss = 0.5287 | Elapsed = 333.8s | ETA = 34998.7s
Epoch 8/10 | Batch 500/47636 | Loss = 0.5077 | Elapsed = 370.9s | ETA = 34968.1s
Epoch 8/10 | Batch 550/47636 | Loss = 0.4996 | Elapsed = 406.5s | ETA = 34803.5s
Epoch 8/10 | Batch 600/47636 | Loss = 0.5158 | Elapsed = 447.9s | ETA =

KeyboardInterrupt: 

In [ ]:
# tokenizer_jsn = json.dumps(tokenizer)

In [ ]:
# with open("tokenizer.json", "w") as f:
#     f.write(tokenizer_jsn)

In [ ]:
generator = Generator(
    model,
    data.tokenizer.stoi,
    data.tokenizer.itos,
    cfg.block_size,
    cfg.get_device(),
)
generated_text = generator.generate_string(
    "python is",
    max_new_tokens=cfg.diagnostic_sample_tokens,
    temperature=0.9,
    repetition_penalty=1.2,
    top_p=0.9,
)
print("\nGenerated sample:")
print(generated_text)
print()
print(build_training_diagnostics_report(model, data, cfg))
if cfg.run_long_context_evaluation:
    results = run_long_context_evaluation(
        cfg,
        block_sizes=cfg.long_context_block_sizes,
        sample_prompt="Programming is",
    )
    print()
    print(format_long_context_results(results))

tensor([ 37, 919])


tensor([  37,  919, 9956])
tensor([  37,  919, 9956, 4194])
tensor([  37,  919, 9956, 4194,  490])
tensor([  37,  919, 9956, 4194,  490,   80])
tensor([  37,  919, 9956, 4194,  490,   80, 9952])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779, 8310])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779, 8310,
        9956])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779, 8310,
        9956, 1544])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779, 8310,
        9956, 1544, 4828])
tensor([  37,  919, 9956, 4194,  490,   80, 9952, 1805, 2098, 9944, 9779, 8310,
        9956, 1544, 4828, 1563])
tensor([  37,  919, 9956, 4194,  490,   80,